<a href="https://colab.research.google.com/github/pra-dyumna/Fine_tune-T5-model-on-finance/blob/main/Actual_T5_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset, DataLoader
import json

# Step 1: Define a Dataset Class
class FinanceDataset(Dataset):
    def __init__(self, data_file):
        self.data = []
        with open(data_file, "r") as f:
            # Load the entire JSON array
            entries = json.load(f)
            for entry in entries:
                task = entry["task"]
                input_text = entry["input"]
                output_text = entry["output"]

                if task == "question_answering":
                    prompt = f"question_answering: {input_text}"
                elif task == "summarization":
                    prompt = f"summarization: {input_text}"
                else:
                    raise ValueError(f"Unknown task: {task}")

                self.data.append({"input": prompt, "output": output_text})

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return item["input"], item["output"]


# Step 2: Tokenize the Data
class TokenizedFinanceDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        input_text, output_text = self.dataset[idx]
        input_enc = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        output_enc = self.tokenizer(
            output_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": input_enc["input_ids"].squeeze(),
            "attention_mask": input_enc["attention_mask"].squeeze(),
            "labels": output_enc["input_ids"].squeeze(),
        }

# Step 3: Load and Prepare Data
def load_data(file_path, tokenizer, max_length=512):
    dataset = FinanceDataset(file_path)
    tokenized_dataset = TokenizedFinanceDataset(dataset, tokenizer, max_length)
    return tokenized_dataset

# Step 4: Fine-Tune the T5 Model
def fine_tune_t5(data_file, output_dir, epochs=15, batch_size=8, learning_rate=5e-5):
    # Load tokenizer and model
    tokenizer = T5Tokenizer.from_pretrained("t5-base")
    model = T5ForConditionalGeneration.from_pretrained("t5-base")

    # Prepare dataset and dataloaders
    dataset = load_data(data_file, tokenizer)
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    # Define training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        logging_dir=f"{output_dir}/logs",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        fp16=torch.cuda.is_available(),
    )

    # Define Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
    )

    # Fine-tune the model
    trainer.train()

    # Save the final model
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model fine-tuned and saved to {output_dir}")

# Step 5: Main Function
if __name__ == "__main__":
    # Path to your dataset file (JSONL format, one example per line)
    data_file = "/content/Taskbase_Data.json"
    # Output directory for the fine-tuned model
    output_dir = "./fine_tuned_t5"
    fine_tune_t5(data_file, output_dir)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-4-48b16b5e5dc2>:103: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,No log,0.080658
2,2.901600,0.044087
3,0.061100,0.037083
4,0.043500,0.034665
5,0.043500,0.032945
6,0.037700,0.032223
7,0.033200,0.031328
8,0.031700,0.031148
9,0.029900,0.030819
10,0.029900,0.030698


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Model fine-tuned and saved to ./fine_tuned_t5


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load fine-tuned model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("./fine_tuned_t5")
tokenizer = T5Tokenizer.from_pretrained("./fine_tuned_t5")


In [ ]:
def generate_response(task, input_text):
    prompt = f"{task}: {input_text}"  # Example: "question_answering: Question: ..."
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    # Generate output
    outputs = model.generate(
        inputs.input_ids,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
task = "question_answering"
input_text = "Question: what are the revenue of US"
response = generate_response(task, input_text)
print("Generated Answer:", response)


Generated Answer: US economy generates a revenue of 11.6 trillion ($24.7 billion) annually.


In [ ]:
task = "summarization"
input_text = "Goldman Sachs is a leading global investment banking, securities, and investment management firm. It provides a wide range of financial services to corporations, financial institutions, governments, and individuals worldwide."
response = generate_response(task, input_text)
print("Generated Summary:", response)


Generated Summary: Goldman Sachs is a leading global investment bank providing diverse financial services.
